**© Copyright AIDENTIFY. All rights reserved.**

# Part 2 | Session 11: 파인튜닝 개념 정리 — SFT/CPT/IT, FFT/PEFT 한눈에 보기

---

### 📋 학습 목표

- 🔹 LLM 학습 파이프라인(Pretraining → SFT → RLHF)을 이해합니다
- 🔹 CPT(Continuous Pretraining)와 IT(Instruction Tuning)의 차이를 파악합니다
- 🔹 FFT(Full Fine-tuning)와 PEFT(Parameter-Efficient FT)의 차이를 비교합니다
- 🔹 LoRA / QLoRA의 핵심 아이디어를 미리 살펴봅니다 (자세한 내용은 Session 12a)
- 🔹 RTX 4060(8GB)에서 학습 가능한 모델 범위를 파악합니다

### 📦 필요 라이브러리

```
numpy (간단한 계산만 — GPU 불필요)
```

### ⏱️ 예상 소요 시간: 약 60분

---

> 💡 본 세션은 **이론/개념 정리**입니다. Part 2 전체(SFT/LoRA/CPT/IT)의 지도를 먼저 그리고 시작합니다.

In [1]:
# 이 노트북은 이론 중심이므로 GPU가 필요하지 않습니다
import numpy as np

print("✅ Session 11: 파인튜닝 개념 정리")
print("📌 GPU 없이 실행 가능 — 간단한 계산과 표로 구성")

✅ Session 11: 파인튜닝 개념 정리
📌 GPU 없이 실행 가능 — 간단한 계산과 표로 구성


---

## 1️⃣ 파인튜닝이란?

**파인튜닝(Fine-tuning)** = 사전학습(Pretraining)된 모델을 **특정 목적·도메인에 맞게 추가 학습**시키는 과정

### 사전학습 모델 vs 파인튜닝 후

| 사전학습 모델 | 파인튜닝 후 |
|---------------|-------------|
| 범용적 언어 이해 | 특정 도메인 전문 지식 |
| 일반적 답변 | 원하는 형식의 응답 |
| 넓지만 얕은 지식 | 좁지만 깊은 전문성 |
| 영어 중심 | 한국어 특화 가능 |

### 비유: 대학 졸업 vs 직무 교육

```
🎓 사전학습 = 대학 교육 (수년, 비싸지만 범용)
🔧 파인튜닝 = 직무 교육 (수일, 저렴하고 특화)
```

→ 파인튜닝은 사전학습보다 훨씬 짧고 저렴한 비용으로 가능합니다.

---

---

## 2️⃣ 학습 3단계: Pretraining → SFT → RLHF

LLM은 일반적으로 **3단계**로 학습됩니다.

### Stage 1: Pretraining (사전학습)
```
📚 인터넷 텍스트 (수조 토큰)
    ↓ Next Token Prediction (자기지도학습)
🧠 Base Model
```

### Stage 2: SFT (Supervised Fine-Tuning)
```
👤 (질문, 답변) 쌍 데이터
    ↓ 지도학습
🤖 Instruction-following Model
```

### Stage 3: RLHF (Reinforcement Learning from Human Feedback)
```
👥 인간 선호도 데이터 (응답 A vs B)
    ↓ PPO / DPO / GRPO
🌟 Aligned Model
```

> 본 Part 2에서는 **Stage 2(SFT)** 에 집중합니다. Stage 3(RLHF)는 Part 3에서 다룹니다.

---

In [2]:
# 단계별 비용/데이터 비교
print("📊 LLM 학습 3단계 비교")
print("=" * 78)

stages = [
    ("1. Pretraining", "1~15조 토큰",   "인터넷 텍스트 (비지도)",     "$1M ~ $100M+", "수십만 GPU-h", "Base Model"),
    ("2. SFT",         "1만 ~ 100만 쌍", "(질문, 답변) 쌍",            "$100 ~ $10K",  "수십 GPU-h",   "Chat Model"),
    ("3. RLHF",        "1만 ~ 10만 쌍",  "선호도 비교 데이터",          "$1K ~ $50K",   "수백 GPU-h",   "Aligned Model"),
]

hdr = f"{'단계':<14} {'데이터 규모':<14} {'데이터 유형':<22} {'비용':<14} {'GPU 시간':<14} {'결과물':<14}"
print(hdr)
print("-" * 78)
for s in stages:
    print(f"{s[0]:<14} {s[1]:<14} {s[2]:<22} {s[3]:<14} {s[4]:<14} {s[5]:<14}")

print("\n💡 우리가 실습할 파인튜닝은 Stage 2 (SFT) — 적은 데이터·비용으로 큰 효과.")

📊 LLM 학습 3단계 비교
단계             데이터 규모         데이터 유형                 비용             GPU 시간         결과물           
------------------------------------------------------------------------------
1. Pretraining 1~15조 토큰       인터넷 텍스트 (비지도)          $1M ~ $100M+   수십만 GPU-h      Base Model    
2. SFT         1만 ~ 100만 쌍    (질문, 답변) 쌍             $100 ~ $10K    수십 GPU-h       Chat Model    
3. RLHF        1만 ~ 10만 쌍     선호도 비교 데이터             $1K ~ $50K     수백 GPU-h       Aligned Model 

💡 우리가 실습할 파인튜닝은 Stage 2 (SFT) — 적은 데이터·비용으로 큰 효과.


---

## 3️⃣ CPT (Continuous Pretraining) 개념

**CPT** = 사전학습된 모델에 **추가 도메인 텍스트를 계속 학습**시키는 것

```
🧠 Base Model (일반 지식)
    ↓ + 의학 논문 100만개
🏥 Medical Model (의학 지식 강화)
```

### 언제 CPT를 쓰나?

| 상황 | CPT 필요성 | 예시 |
|------|------------|------|
| 도메인 전문 용어가 많은 경우 | ⭐⭐⭐ | 의학, 법률, 금융 |
| 한국어 성능 강화 | ⭐⭐⭐ | 영어 기반 모델 → 한국어 |
| 최신 정보 반영 | ⭐⭐ | 학습 컷오프 이후 뉴스 |
| 일반적 작업 | ⭐ | SFT만으로 충분 |

### CPT vs SFT 핵심 차이

| 항목 | CPT | SFT |
|------|-----|-----|
| 목적 | 도메인 **지식** 습득 | **지시 수행** 능력 |
| 데이터 형식 | 일반 텍스트 (비구조화) | (질문, 답변) 쌍 |
| 학습 방법 | Next Token Prediction | Next Token Prediction |
| 데이터 양 | 많을수록 좋음 (수억 토큰) | 수만~수십만 개 |
| 학습률 | 매우 낮음 (≈1e-5) | 약간 높음 (1e-4~3e-4) |

> 자세한 실습은 **Session 15: Continuous Pretraining** 에서.

---

---

## 4️⃣ Instruction Tuning (IT) 개념

**IT** = 모델이 사용자의 **지시(instruction)를 이해하고 따르도록** 학습시키는 것.

### SFT vs Instruction Tuning

사실 둘은 **거의 같은 개념**으로 사용됩니다.
- 📌 **SFT** = 학습 방법론 (지도학습 기반 파인튜닝)
- 📌 **Instruction Tuning** = SFT의 목적/효과 (지시 수행 능력 부여)

### Instruction 데이터 형식

```json
{
    "instruction": "다음 문장을 영어로 번역해주세요.",
    "input": "오늘 날씨가 좋습니다.",
    "output": "The weather is nice today."
}
```

### Instruction 유형 예시

| 유형 | 예시 |
|------|------|
| 요약 | "다음 기사를 3줄로 요약해줘" |
| 번역 | "한국어를 영어로 번역해줘" |
| 분류 | "이 리뷰의 감성을 분석해줘" |
| 생성 | "키워드로 이메일을 작성해줘" |
| 질의응답 | "제2차 세계대전은 언제 끝났어?" |
| 코딩 | "Python으로 정렬 알고리즘을 구현해줘" |

> 자세한 실습은 **Session 16: Instruction Tuning** 에서.

---

---

## 5️⃣ Full Fine-tuning vs Parameter-Efficient Fine-tuning (PEFT)

### Full Fine-tuning (FFT)
모델의 **모든 파라미터**를 업데이트하는 방식.

```
전체 파라미터 ✅ 업데이트
장점: 최고 성능 가능
단점: 매우 많은 VRAM
```

### Parameter-Efficient Fine-tuning (PEFT)
모델의 **일부 파라미터만** 업데이트.

```
기존 파라미터 ❄️ 동결 (Frozen)
추가 파라미터 ✅ 학습 (매우 적은 수)
장점: 적은 VRAM
```

### 비교

| 항목 | FFT | PEFT (LoRA) |
|------|-----|-------------|
| 학습 파라미터 | 100% | 0.1~5% |
| VRAM 사용량 | 매우 높음 | 낮음 |
| 성능 | 최고 | FFT의 90~99% |
| 학습 속도 | 느림 | 빠름 |
| RTX 4060 가능? | 1.5B까지만 | 7B까지 (QLoRA) |

---

In [3]:
# FFT vs PEFT VRAM 사용량 비교 (추정)
def calculate_vram(params_b, method="fft", precision="fp16"):
    bytes_per_param = 2 if precision == "fp16" else 4

    model_size_gb = params_b * 1e9 * bytes_per_param / 1e9

    if method == "fft":
        optimizer_gb = params_b * 1e9 * 4 * 2 / 1e9
        gradient_gb = model_size_gb
        activation_gb = model_size_gb * 0.3
        total_gb = model_size_gb + optimizer_gb + gradient_gb + activation_gb
    elif method == "lora":
        lora_ratio = 0.02
        lora_params = params_b * lora_ratio
        optimizer_gb = lora_params * 1e9 * 4 * 2 / 1e9
        gradient_gb = lora_params * 1e9 * bytes_per_param / 1e9
        activation_gb = model_size_gb * 0.2
        total_gb = model_size_gb + optimizer_gb + gradient_gb + activation_gb
    elif method == "qlora":
        model_size_gb = params_b * 1e9 * 0.5 / 1e9
        lora_ratio = 0.02
        lora_params = params_b * lora_ratio
        optimizer_gb = lora_params * 1e9 * 4 * 2 / 1e9
        gradient_gb = lora_params * 1e9 * 2 / 1e9
        activation_gb = model_size_gb * 0.3
        total_gb = model_size_gb + optimizer_gb + gradient_gb + activation_gb

    return total_gb


print("📊 FFT vs LoRA vs QLoRA VRAM 추정 (RTX 4060 = 8GB 기준)")
print("=" * 95)

models = [1.5, 3, 7, 13, 70]
methods = ["fft", "lora", "qlora"]
labels = ["FFT (fp16)", "LoRA (fp16)", "QLoRA (4bit)"]

print(f"{'모델':>8s}", end="")
for lbl in labels:
    print(f" | {lbl:>22s}", end="")
print()
print("-" * 95)

for m in models:
    print(f"{m:>6.1f}B", end="")
    for method in methods:
        vram = calculate_vram(m, method)
        mark = "✅" if vram <= 8 else "❌"
        print(f" | {vram:>16.1f}GB {mark:>4s}", end="")
    print()

print("\n📌 RTX 4060 (8GB) 결론:")
print("   - FFT:   1.5B까지만 가능")
print("   - LoRA:  3B까지 가능")
print("   - QLoRA: 7B까지 빠듯하게 가능")
print("\n⚠️ 실제 VRAM은 시퀀스 길이/배치 크기에 따라 달라집니다.")

📊 FFT vs LoRA vs QLoRA VRAM 추정 (RTX 4060 = 8GB 기준)
      모델 |             FFT (fp16) |            LoRA (fp16) |           QLoRA (4bit)
-----------------------------------------------------------------------------------------------
   1.5B |             18.9GB    ❌ |              3.9GB    ✅ |              1.3GB    ✅
   3.0B |             37.8GB    ❌ |              7.8GB    ✅ |              2.5GB    ✅
   7.0B |             88.2GB    ❌ |             18.2GB    ❌ |              6.0GB    ✅
  13.0B |            163.8GB    ❌ |             33.8GB    ❌ |             11.0GB    ❌
  70.0B |            882.0GB    ❌ |            182.0GB    ❌ |             59.5GB    ❌

📌 RTX 4060 (8GB) 결론:
   - FFT:   1.5B까지만 가능
   - LoRA:  3B까지 가능
   - QLoRA: 7B까지 빠듯하게 가능

⚠️ 실제 VRAM은 시퀀스 길이/배치 크기에 따라 달라집니다.


---

## 6️⃣ LoRA 핵심 아이디어 (미리보기)

> 자세한 수학과 실습은 **Session 12a: LoRA 이론** 에서.

### LoRA (Low-Rank Adaptation) 한 줄 요약

기존 가중치 $W$를 직접 수정하는 대신, **저차원 행렬 2개의 곱($BA$)** 으로 변화량을 표현.

$$W' = W + \Delta W = W + (\alpha / r) \cdot BA$$

- $W \in \mathbb{R}^{d \times k}$ : 원본 (동결 ❄️)
- $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times k}$ : 학습 🔥
- $r$ : 랭크 (8, 16, 32, 64)
- $\alpha$ : 스케일링 (보통 $r$ 또는 $2r$)

### 파라미터 수 비교 (d=k=4096, r=16)

```
원본 W : 4096 × 4096 = 16,777,216 (1677만)
LoRA   : 16 × (4096 + 4096) = 131,072 (13만)
비율   : 0.78%   ← 원본의 1% 미만!
```

→ **0.5~1% 파라미터만 학습**해도 FFT의 90~99% 성능.

---

---

## 7️⃣ QLoRA 핵심 아이디어 (미리보기)

> 자세한 양자화 이론과 실습은 **Session 23~24** 에서.

### QLoRA = 4bit 양자화된 모델 위에서 LoRA

**3가지 기술**:
1. **NF4 (4-bit NormalFloat)** — 정규분포에 최적화된 4비트 데이터 타입
2. **Double Quantization** — 양자화 상수까지 다시 양자화 (≈0.37 bit/param 추가 절약)
3. **Paged Optimizers** — OOM 방지용 CPU 스왑

### 메모리 비교 (7B 모델 기준)

| 방법 | 모델 메모리 | 총 VRAM | RTX 4060 |
|------|-------------|---------|----------|
| FFT (fp16) | 14GB | ~56GB | ❌ |
| LoRA (fp16) | 14GB | ~18GB | ❌ |
| **QLoRA (4bit)** | **3.5GB** | **~6-8GB** | ✅ |

→ QLoRA 덕분에 7B 모델도 8GB GPU에서 파인튜닝 가능.

---

In [4]:
# 양자화 비트별 모델 메모리만 비교 (학습 오버헤드 제외)
def model_memory_gb(params_b, bits):
    return params_b * (bits / 8)

print("📊 양자화 비트별 모델 로드 메모리")
print("=" * 78)

model_sizes = [1.5, 3, 7, 13, 70]
bit_configs = [(32, "FP32"), (16, "FP16/BF16"), (8, "INT8"), (4, "NF4 (QLoRA)")]

print(f"{'모델':>8s}", end="")
for _, name in bit_configs:
    print(f" | {name:>14s}", end="")
print()
print("-" * 78)

for size in model_sizes:
    print(f"{size:>6.1f}B", end="")
    for bits, _ in bit_configs:
        mem = model_memory_gb(size, bits)
        fit = "✅" if mem <= 8 else "❌"
        print(f" | {mem:>10.1f}GB {fit:>2s}", end="")
    print()

print("\n📌 8GB(RTX 4060) 기준 ✅ 표시")
print("💡 QLoRA(NF4) → 7B 모델도 3.5GB로 로드 가능")

📊 양자화 비트별 모델 로드 메모리
      모델 |           FP32 |      FP16/BF16 |           INT8 |    NF4 (QLoRA)
------------------------------------------------------------------------------
   1.5B |        6.0GB  ✅ |        3.0GB  ✅ |        1.5GB  ✅ |        0.8GB  ✅
   3.0B |       12.0GB  ❌ |        6.0GB  ✅ |        3.0GB  ✅ |        1.5GB  ✅
   7.0B |       28.0GB  ❌ |       14.0GB  ❌ |        7.0GB  ✅ |        3.5GB  ✅
  13.0B |       52.0GB  ❌ |       26.0GB  ❌ |       13.0GB  ❌ |        6.5GB  ✅
  70.0B |      280.0GB  ❌ |      140.0GB  ❌ |       70.0GB  ❌ |       35.0GB  ❌

📌 8GB(RTX 4060) 기준 ✅ 표시
💡 QLoRA(NF4) → 7B 모델도 3.5GB로 로드 가능


---

## 8️⃣ RTX 4060 (8GB) 학습 가능 범위

### 모델 크기별 가능 방식

| 모델 | FFT | LoRA (fp16) | QLoRA (4bit) | 추론만 |
|------|-----|-------------|---------------|--------|
| 0.5B | ✅ | ✅ | ✅ | ✅ |
| **1.5B** | ✅ (빠듯) | ✅ | ✅ | ✅ |
| **3B** | ❌ | ✅ (빠듯) | ✅ | ✅ |
| **7B** | ❌ | ❌ | ✅ (빠듯) | ✅ (4bit) |
| 13B | ❌ | ❌ | ❌ | ❌ |

### 본 과정 실습 모델 권장

| 용도 | 권장 모델 | 방법 |
|------|-----------|------|
| FFT 실습 | Qwen2.5-1.5B-Instruct | FFT (fp16) |
| LoRA 실습 | Qwen2.5-1.5B-Instruct | LoRA (r=16, alpha=32) |
| LoRA vs FFT 비교 | Qwen2.5-1.5B-Instruct | 둘 다 가능한 크기 |
| QLoRA 실습 (Part 3) | Qwen2.5-7B-Instruct | QLoRA (4bit NF4) |

---

In [5]:
# RTX 4060 최적 설정 가이드
configs = [
    {
        "name": "Qwen2.5-1.5B + FFT",
        "method": "Full Fine-tuning",
        "precision": "fp16",
        "batch_size": 1, "grad_accum": 8, "max_seq_len": 1024,
        "vram_est": "~7GB", "note": "VRAM 빠듯 — 시퀀스 길이 주의",
    },
    {
        "name": "Qwen2.5-1.5B + LoRA",
        "method": "LoRA (r=16, alpha=32)",
        "precision": "fp16",
        "batch_size": 2, "grad_accum": 4, "max_seq_len": 2048,
        "vram_est": "~5GB", "note": "여유 — 권장 기본 설정",
    },
    {
        "name": "Qwen2.5-3B + LoRA",
        "method": "LoRA (r=16, alpha=32)",
        "precision": "fp16",
        "batch_size": 1, "grad_accum": 8, "max_seq_len": 1024,
        "vram_est": "~7GB", "note": "VRAM 빠듯",
    },
    {
        "name": "Qwen2.5-7B + QLoRA",
        "method": "QLoRA (4bit NF4)",
        "precision": "4bit + fp16 compute",
        "batch_size": 1, "grad_accum": 8, "max_seq_len": 1024,
        "vram_est": "~7GB", "note": "VRAM 빠듯 — Unsloth 사용 권장",
    },
]

print("📋 RTX 4060 (8GB) 최적 학습 설정 가이드")
print("=" * 78)
for c in configs:
    print(f"\n🏷️  {c['name']}")
    print(f"  학습 방법:       {c['method']}")
    print(f"  정밀도:          {c['precision']}")
    print(f"  배치 / 누적:     {c['batch_size']} × {c['grad_accum']}")
    print(f"  최대 시퀀스:     {c['max_seq_len']}")
    print(f"  예상 VRAM:       {c['vram_est']}")
    print(f"  비고:            {c['note']}")

print("\n\n💡 권장 기본:")
print("   - 실습 베이스: Qwen2.5-1.5B-Instruct")
print("   - 모든 실습: fp16=True (메모리 절약)")
print("   - QLoRA 시: Unsloth 활용 권장 (Session 23/24)")

📋 RTX 4060 (8GB) 최적 학습 설정 가이드

🏷️  Qwen2.5-1.5B + FFT
  학습 방법:       Full Fine-tuning
  정밀도:          fp16
  배치 / 누적:     1 × 8
  최대 시퀀스:     1024
  예상 VRAM:       ~7GB
  비고:            VRAM 빠듯 — 시퀀스 길이 주의

🏷️  Qwen2.5-1.5B + LoRA
  학습 방법:       LoRA (r=16, alpha=32)
  정밀도:          fp16
  배치 / 누적:     2 × 4
  최대 시퀀스:     2048
  예상 VRAM:       ~5GB
  비고:            여유 — 권장 기본 설정

🏷️  Qwen2.5-3B + LoRA
  학습 방법:       LoRA (r=16, alpha=32)
  정밀도:          fp16
  배치 / 누적:     1 × 8
  최대 시퀀스:     1024
  예상 VRAM:       ~7GB
  비고:            VRAM 빠듯

🏷️  Qwen2.5-7B + QLoRA
  학습 방법:       QLoRA (4bit NF4)
  정밀도:          4bit + fp16 compute
  배치 / 누적:     1 × 8
  최대 시퀀스:     1024
  예상 VRAM:       ~7GB
  비고:            VRAM 빠듯 — Unsloth 사용 권장


💡 권장 기본:
   - 실습 베이스: Qwen2.5-1.5B-Instruct
   - 모든 실습: fp16=True (메모리 절약)
   - QLoRA 시: Unsloth 활용 권장 (Session 23/24)


---

## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 핵심 개념

| # | 개념 | 핵심 내용 |
|---|------|----------|
| 1️⃣ | 파인튜닝 | 사전학습 모델을 특정 목적에 맞게 추가 학습 |
| 2️⃣ | 학습 3단계 | Pretraining → SFT → RLHF |
| 3️⃣ | CPT | 도메인 텍스트로 지식 보강 (비구조화) |
| 4️⃣ | Instruction Tuning | (질문, 답변) 쌍으로 지시 수행 능력 |
| 5️⃣ | FFT vs PEFT | 전체 vs 일부 파라미터 학습 |
| 6️⃣ | LoRA | 저차원 행렬 분해 (파라미터 ~1%) |
| 7️⃣ | QLoRA | 4bit 양자화 + LoRA (7B도 8GB OK) |
| 8️⃣ | RTX 4060 범위 | FFT: 1.5B / LoRA: 3B / QLoRA: 7B |

### 핵심 수식 요약

```
LoRA:  W' = W + (α/r) × BA
파라미터 수: r × (d + k)  ≪  d × k

QLoRA VRAM ≈ params × 0.5 bytes + LoRA overhead
```

### 다음 세션 예고
- 🔜 **Session 11a**: HuggingFace Transformers & TRL 기본 사용법
- 🔜 **Session 12a**: LoRA vs Full Fine-tuning 이론 비교
- 🔜 **Session 12b**: LoRA vs FFT 실전 비교 실습
- 🔜 **Session 12c**: Unsloth 기반 파인튜닝 (LoRA 가속)
- 🔜 **Session 13**: 데이터 파이프라인 & 합성 데이터/Distillation
- 🔜 **Session 14**: 본격 SFT 실습 — TRL SFTTrainer + LoRA + Qwen2.5-1.5B

---

### 💡 퀴즈 (다음 세션 시작 시 확인)
1. 7B 모델을 RTX 4060에서 학습하려면 어떤 방법을 써야 할까요?
2. LoRA의 rank가 16이고 hidden dim이 4096일 때, 원본 대비 몇 %의 파라미터를 학습할까요?
3. CPT와 SFT의 가장 큰 차이는?

---

In [6]:
print("✅ Session 11 완료!")
print("📚 다음 세션: SFT 실습 (HuggingFace Trainer / TRL SFTTrainer)")

✅ Session 11 완료!
📚 다음 세션: SFT 실습 (HuggingFace Trainer / TRL SFTTrainer)
